In [1]:
#imports
import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

In [2]:
#connect to google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
data = np.load("/content/drive/MyDrive/AML/AML_resnet_features.npz",allow_pickle=True)
clinical_df=pd.read_csv('/content/drive/MyDrive/AML/cleaned_clinical_data.csv')

X_img = data["X_img"]
y_img = data["y_img"]
pids_img = data["pids_img"]

In [4]:
def aggregate_patient_features(X, patient_ids, labels):
    df = pd.DataFrame(X)
    df["patient_id"] = patient_ids
    df["target"] = labels

    feature_cols = df.columns[:-2]   # only image feature columns

    grouped_X = df.groupby("patient_id")[feature_cols].mean()
    grouped_y = df.groupby("patient_id")["target"].first()

    X_out = grouped_X.values
    y_out = grouped_y.values.astype(int)
    pids_out = grouped_X.index.values

    return X_out, y_out, pids_out

In [5]:
X_img_patient, y_patient, patient_ids = aggregate_patient_features(X_img, pids_img, y_img)
print(X_img_patient.shape)

(189, 2048)


In [6]:
clinical_df = clinical_df.set_index("patient_id").loc[patient_ids].reset_index()

X_clinical = clinical_df.drop(columns=["patient_id", "target_multi"]).values

In [7]:
scaler_img = StandardScaler()
X_img_scaled = scaler_img.fit_transform(X_img_patient)

pca = PCA(n_components=0.95)
X_img_pca = pca.fit_transform(X_img_scaled)

In [8]:
scaler_clinical = StandardScaler()
X_clinical_scaled = scaler_clinical.fit_transform(X_clinical)

In [9]:
X_full = np.concatenate([X_img_pca, X_clinical_scaled], axis=1)
y_full = y_patient

In [10]:
best_xgb_params = {
    "objective": "multi:softprob",
    "num_class": 5,
    "n_estimators": 500,
    "max_depth": 5,
    "learning_rate": 0.03,
    "subsample": 0.6,
    "colsample_bytree": 0.6,
    "gamma": 0.5,
    "min_child_weight": 3,
    "random_state": 42,
    "eval_metric": "mlogloss"
}

In [11]:
sample_weights = compute_sample_weight(class_weight="balanced", y=y_full)

final_model = XGBClassifier(**best_xgb_params)

final_model.fit(X_full, y_full, sample_weight=sample_weights)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.6, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=0.5,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.03, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=5, max_leaves=None,
              min_child_weight=3, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None, num_class=5, ...)

In [12]:
preprocess_objects = {
    "scaler_img": scaler_img,
    "pca": pca,
    "scaler_clinical": scaler_clinical,
    "clinical_feature_names": list(clinical_df.drop(columns=["patient_id","target_multi"]).columns),
    "label_map": {0:"Control",1:"NPM1",2:"PML_RARA",3:"RUNX1_RUNX1T1",4:"CBFB_MYH11"},
    "backbone": "ResNet50"
}

In [13]:
SAVE_PATH = "/content/drive/MyDrive/AML/"

joblib.dump(final_model, SAVE_PATH + "final_xgboost_model.pkl")
joblib.dump(preprocess_objects, SAVE_PATH + "preprocessing.pkl")

['/content/drive/MyDrive/AML/preprocessing.pkl']